In [ ]:
!pip install google-genai -q

In [ ]:
import google.genai as genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_KEY'))

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello in one sentence"
)
print(response.text)

Hello there!


In [ ]:
import pandas as pd
df = pd.read_csv("hr_test_3employees.csv")
print(df.shape)
print(df.head())

(3, 8)
  employee_id  overtime_hours  leave_days  participation_rate  \
0     EMP_001              42           0                  28   
1     EMP_016              28           3                  58   
2     EMP_031              22           6                  80   

   output_consistency  tenure_months  \
0                  45             18   
1                  72             24   
2                  90             24   

                                     survey_response risk_category  
0  I am completely exhausted and struggling to ke...          high  
1  Things have been a bit hectic lately but I am ...        medium  
2  I really enjoy my work and feel well supported...           low  


In [ ]:
import time


def assess_burnout_risk(row):
  prompt = f"""
You are an HR wellbeing analyst reviewing anonymised employee data.
Your job is to assess burnout risk based on the data provided.

Respond in exactly this format:
RISK LEVEL: [high/medium/low]
KEY SIGNALS: [list the specific data points that drove your assessment]
EXPLANATION: [2 to 3 sentences explaining your reasoning in plain language]
CONFIDENCE: [high/medium/low]

Do not make assumptions beyond what the data shows.

Employee data:
-Employee ID: {row['employee_id']}
-Overtime hours this month: {row['overtime_hours']}
-Leave days taken this quarter: {row['leave_days']}
-Team partipation rate: {row['participation_rate']}
-Output consistency score: {row['output_consistency']}
-Tenure in current role (months): {row['tenure_months']}
-Survey response: {row['survey_response']}
"""
  max_retries = 5
  retry_count = 0

  while retry_count < max_retries:
    try:
     result = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
     )
     time.sleep(15)
     return result.text
    except Exception as e:
      error_message = str(e)
      retry_count += 1
      if "429" in error_message:
        print(f"Rate limit hit.Waiting 60 seconds before retry {retry_count} of {max_retries}...")
        time.sleep(60)
      elif "503" in error_message:
        print(f"Service unavailable. Waiting 30 seconds before retry {retry_count} of {max_retries}...")
        time.sleep(30)
      else:
        print(f"Unexpected error: {error_message}")
        time.sleep(15)

  print(f"Failed after {max_retries} retries for {row['employee_id']} ")
  return "ASSESSMENT FAILED"

In [ ]:
df_small = df.head(3)
results = []

for index, row in df_small.iterrows():
  print(f"Processing {row['employee_id']}...")
  assessment = assess_burnout_risk(row)
  results.append({"employee_id": row['employee_id'], "assessment": assessment})
  print(f"Done: {row['employee_id']}")

print("All done")
print(f"Total results: {len(results)}")

Processing EMP_001...
Done: EMP_001
Processing EMP_016...
Done: EMP_016
Processing EMP_031...
Done: EMP_031
All done
Total results: 3


In [ ]:
for r in results:
  print(f"\n{'='*50}")
  print(f"Employee: {r['employee_id']}")
  print(f"{'='*50}")
  print(r['assessment'])


Employee: EMP_001
RISK LEVEL: high
KEY SIGNALS: Overtime hours this month (42), Leave days taken this quarter (0), Team participation rate (28), Output consistency score (45), Survey response ("I am completely exhausted and struggling to keep up with the workload. I feel like I am drowning and nobody notices.")
EXPLANATION: The employee is experiencing extremely high overtime and has taken no leave, indicating severe overwork and lack of rest. This objective data aligns perfectly with their explicit survey response detailing complete exhaustion, feeling overwhelmed by workload, and a sense of unacknowledgment.
CONFIDENCE: high

Employee: EMP_016
RISK LEVEL: Medium
KEY SIGNALS: Overtime hours this month (28), Survey response ("Things have been a bit hectic lately but I am managing. Some weeks are harder than others"), Team participation rate (58)
EXPLANATION: This employee is working significant overtime and acknowledges a "hectic" workload with "harder" weeks, suggesting elevated stre

In [ ]:
import pandas as pd
results_df = pd.DataFrame(results)
results_df.to_csv("results_output.csv", index=False)
print("Results saved successfully")

Results saved successfully


In [ ]:
app_code = '''
import streamlit as st
import pandas as pd
import time
from google import genai

# Setup
api_key = "paste-your-api-key-here"
client = genai.Client(api_key=api_key)

def assess_burnour_risk(row):
  prompt = f"Analyse this employee for burnout risk. Employee ID: {row\'employee_id\'}, Overtime hours: {row[\'overtime_hours\']}, Leave days:{row[\'leave_days\']}, Participation rate: {row[\'participation_rate\']}, Output consistency: {row[\'output_consistency\']}, Tenure months: {row[\'tenure_months\']}, Survey response: {row[\'survey_response\']}". Respond with RISK LEVEL: high/medium/low, KEY SIGNALS: list signals, EXPLANATION: 2 sentences, CONFIDENCE: high/medium/low"
  try:
    result = client.models.generate_content(model="gemini-2.5-flash, contents=prompt)
    time.sleep(15)
    return result.text
  except Exception as e:
    return f"ERROR = str(e)"
def get_risk_level(assessment):
  for line in assessment.split("\n"):
    if line.startswith("RISK LEVEL:"):
       if "RISK LEVEL:" in line.upper():
        text = line.split(":")[-1].strip().lower()
        if "high" in text:
          return "high"
        elif "medium" in text:
          returm "medium"
        else:
          return "low"
 return "unknown"

# App
st.set_page_config(page_title="HR Wellbeing System", layout="wide")
st.title("HR Wellbeing System")
st.caption("AI powered burnout risk and wellbeing analysis")

# Role selector
st.sidebar.title("User Access)
role = st.sidebar.selectbox("Select your role", ["People Partner", "Team Lead", "Employee", "Compliance Officer])
st.sidebar.info(f"Logged in as: {role}")

uploaded_file = st.file_uploader("Upload workfoce CSV", type=["csv"])

if uploaded_file:
  df = pd.read_csv(uploaded_file)
  st.success(f"File uploaded. {len(df)} employee records found.")

  if st.button("Run Analysis"):
    results = []
    with st.spinner("Analysing workforce data..."):
      for index, row in df.iterrows():
        assessment = assess_burnout_risk(row)
        risk_level = get_risk_level(assessment)
        results.append({
          "employee_id": row["employee_id"],
          "risk_level": risk,
          "assessment": assessment
        })
  results_df = pd.DataFrame(results)

  if role == "People Partner":
    st.subheader("Workforce Wellbeing Report")
    for _, r in results_df.iterrows():
      if r["risk_level"] == "high":
        st.error(f"{r[\'employee_id\']}: HIGH RISK")
      elif r["risk_level"] == "medium":
        st.warning(f"{r[\'employee_id\']}: MEDIUM RISK")
      else:
        st.success(f"{r[\'employee_id\']}: LOW RISK")
      st.write(r["assessment"])
      st.divider()

elif role == "Team Lead":
  st.subheader("Team Wellbeing Dashboard")
  st.info("Showing aggregated team data only. Individual identities are protected.")
  high = len(results_df[results_df["risk_level"] == "high"])
  medium = len(results_df[results_df["risk_level"] == "medium"])
  low = len(results_df[results_df["risk_level"] == "low"])
  col1, col2, col3 = st.columns(3)
  col1.metric("High Risk", high)
  col2.metric("Medium Risk", medium)
  col3.metric("Low Risk", low)
  st.bar_chart(results_df["risk_level"].value_counts())

elif role == "Employee":
  st.subheader("Your Personal Wellbeing Summary")
  st.info("This view is private. Your manager cannot see this information.")
  first = results_df.iloc[0]
  if first["risk_level"] == "high":
    st.error("Your current working patterns suggest you may be at risk of burnout.")
  elif first["risk_level"] == "medium":
    st.warning("Your current working patterns show some signs of strain worth monitoring.")
  else:
    st.success("Your current working patterns are in good shape.")
  st.write(first["assessment"])
  if st.button("Flag inaccurate summary"):
    st.success("Your feedback has been recorded.")

elif role == "Compliance Officer":
  st.subheader("Audit and Compliance Dashboard")
  st.info("System level outputs only. Individual identities are protected.")
  st.dataframe(results_df[["employee_id", "risk_level"]])
  st.subheader("Risk Distribution")
  st.bar_chart(results_df["risk_level"].value_counts())

'''

with open("app_code.py", "w") as f:
  f.write(app_code)

print("App code saved successfully")




App code saved successfully


In [ ]:
import subprocess
import threading
import time

def run_streamlit():
  subprocess.run(["streamlit", "run", "/content/app_code.py", "--server.port=8501", "--server.headless=true"])
thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

time.sleep(5)

from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(8501)"))

https://8501-m-s-kkb-euw4b0-2s3xlvaklb4cu-b.europe-west4-0.prod.colab.dev
